# Comparing anomaly scores on a labeled healthcare benchmark

The eICU demo has no verified data-error labels. This notebook adds a separate check using the labeled `annthyroid` healthcare dataset from ADBench. Isolation Forest and an autoencoder are fitted without labels; benchmark labels are used afterward to calculate evaluation metrics.

This answers a narrower question: how well do the scores rank the benchmark's labeled anomalies? It does not establish how well they identify errors in eICU or a health registry.

## Load the benchmark data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support

DATA_PATH = Path("data/adbench/2_annthyroid.npz")
if not DATA_PATH.exists():
    raise FileNotFoundError("Download 2_annthyroid.npz from the official ADBench repository and save it to data/adbench/.")
data = np.load(DATA_PATH, allow_pickle=True)
X_raw = pd.DataFrame(data["X"]).apply(pd.to_numeric, errors="coerce")
y = np.asarray(data["y"]).astype(int).ravel()
print("Features:", X_raw.shape, "Anomaly share:", round(float(y.mean()), 4))
display(X_raw.describe().T)

## Fit both models without labels

The features are imputed and scaled. The autoencoder learns to reconstruct each feature row; its mean squared reconstruction error is used as the anomaly score. Higher scores mean more unusual for both methods.

In [ ]:
imputer = SimpleImputer(strategy="median")
X_imp = imputer.fit_transform(X_raw)
X = RobustScaler().fit_transform(X_imp)

isolation_forest = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
isolation_forest.fit(X)
if_score = -isolation_forest.decision_function(X)

hidden_width = max(2, min(8, X.shape[1] // 2))
autoencoder = MLPRegressor(hidden_layer_sizes=(hidden_width,), activation="relu",
                           solver="adam", max_iter=500, early_stopping=True,
                           random_state=42)
autoencoder.fit(X, X)
ae_score = np.mean((X - autoencoder.predict(X)) ** 2, axis=1)

def score_summary(name, scores):
    cutoff = np.quantile(scores, 0.95)  # same 5% review capacity for both methods
    pred = (scores >= cutoff).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y, pred, average="binary", zero_division=0
    )
    return {"method": name,
            "ROC-AUC": roc_auc_score(y, scores),
            "Average precision": average_precision_score(y, scores),
            "Precision at 5% flagged": precision,
            "Recall at 5% flagged": recall,
            "F1 at 5% flagged": f1}

metrics = pd.DataFrame([
    score_summary("Isolation Forest", if_score),
    score_summary("Autoencoder", ae_score)
])
display(metrics)

## Explain the findings

Discuss ROC-AUC and average precision first; they measure how well the scores rank the labeled anomalies across thresholds. The 5% threshold metrics show one fixed review-capacity choice. State that the benchmark's anomaly labels reflect its source dataset and task definition, which may not match data entry errors in a health registry. Do not merge these numbers with the eICU synthetic-corruption results.

**Source:** ADBench (NeurIPS 2022), official code and data repository: https://github.com/Minqi824/ADBench . Its README lists `2_annthyroid` as a healthcare dataset.